# Qwen2.5-1.5B Finetuning — Middle School Tutor
QLoRA finetuning via Unsloth on T4 GPU.  
**Before running:** upload `training_data.jsonl` to this Colab session.

In [ ]:
# Install dependencies
!pip install unsloth trl transformers datasets accelerate -q

In [ ]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

BASE_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"
DATA_PATH    = "training_data.jsonl"
LORA_DIR     = "qwen-tutor-lora"
OUTPUT_DIR   = "qwen-tutor-merged"
MAX_SEQ_LEN  = 512

SYSTEM_PROMPT = (
    "You are a middle school tutor. For every answer, structure your response "
    "with an EXAMPLE section and a WHY IT MATTERS section, each as bullet points."
)

In [ ]:
# Load dataset
def load_records(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

records = load_records(DATA_PATH)
print(f"Loaded {len(records)} records")
print("Sample:", records[0])

In [ ]:
# Load base model in 4-bit and attach LoRA adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Format records into ChatML and build dataset
def format_chat(record):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": record["question"]},
        {"role": "assistant", "content": record["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

texts  = [format_chat(r) for r in records]
dataset = Dataset.from_dict({"text": texts})
splits  = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(splits['train'])}  Eval: {len(splits['test'])}")

In [ ]:
# Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=splits["train"],
    eval_dataset=splits["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        output_dir=LORA_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
    ),
)

trainer.train()

In [ ]:
# Merge LoRA adapters into base weights and save as full HF model
model.save_pretrained_merged(OUTPUT_DIR, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {OUTPUT_DIR}/")

## Quantize to GGUF (Q4_K_M)
Converts the merged model to a GGUF file you can load in Ollama.

In [ ]:
# Clone and build llama.cpp
!git clone https://github.com/ggerganov/llama.cpp --depth=1
!pip install -r llama.cpp/requirements.txt -q
!cmake -B llama.cpp/build llama.cpp && cmake --build llama.cpp/build --config Release -j$(nproc)

In [ ]:
# Convert merged HF model to GGUF (f16 intermediate)
!python llama.cpp/convert_hf_to_gguf.py qwen-tutor-merged \
    --outtype f16 \
    --outfile qwen-tutor-f16.gguf

In [ ]:
# Quantize to Q4_K_M
!./llama.cpp/build/bin/llama-quantize qwen-tutor-f16.gguf qwen-tutor-q4km.gguf Q4_K_M
!ls -lh qwen-tutor-q4km.gguf

## Done
Download `qwen-tutor-q4km.gguf` from the Colab file browser.  
Place it at `offline_chatbot/models/qwen-tutor-q4km.gguf`, then load into Ollama:
```
ollama create tb-tutor -f llm\Modelfile
```